# 14. Active Localization과 Exploration

Exploration은 단순히 목표까지 가는 게 아니라 **정보를 얻는 행동**을 선택한다.
Occupancy grid에서는 frontier, localization에서는 expected entropy reduction을 자주 쓴다.

$$H(b)=-\sum_x b(x)\log b(x)$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import os
os.makedirs('assets', exist_ok=True)

for font_name in ['Nanum Gothic', 'AppleGothic', 'Malgun Gothic']:
    if any(font.name == font_name for font in fm.fontManager.ttflist):
        plt.rcParams['font.family'] = font_name
        break
plt.rcParams['axes.unicode_minus'] = False

## 1. Frontier 기반 Exploration

알려진 free cell과 unknown cell의 경계가 frontier다. 로봇은 frontier를 향해 움직이며 지도를 확장한다.

In [ ]:
np.random.seed(26)
H,W=40,55
true=np.zeros((H,W),dtype=int) # 0 free, 1 occ
true[0,:]=true[-1,:]=true[:,0]=true[:,-1]=1
true[10:32,18]=1; true[8:22,36]=1; true[27,22:45]=1
known=np.full((H,W),-1) # -1 unknown, 0 free, 1 occ
robot=(30,8)

def reveal(pos,radius=6):
    rr,cc=pos
    for r in range(max(0,rr-radius),min(H,rr+radius+1)):
        for c in range(max(0,cc-radius),min(W,cc+radius+1)):
            if (r-rr)**2+(c-cc)**2<=radius**2:
                known[r,c]=true[r,c]
def frontiers():
    fs=[]
    for r in range(1,H-1):
        for c in range(1,W-1):
            if known[r,c]==0 and any(known[r+dr,c+dc]==-1 for dr,dc in [(-1,0),(1,0),(0,-1),(0,1)]):
                fs.append((r,c))
    return np.array(fs)

def step_toward(pos,target):
    r,c=pos; tr,tc=target
    candidates=[(r+np.sign(tr-r),c),(r,c+np.sign(tc-c)),(r,c)]
    candidates=[(int(a),int(b)) for a,b in candidates]
    valid=[p for p in candidates if true[p]==0]
    return min(valid,key=lambda p:(p[0]-tr)**2+(p[1]-tc)**2)

path=[robot]
for _ in range(45):
    reveal(robot)
    fs=frontiers()
    if len(fs)==0: break
    # score = information frontier count nearby - travel distance
    scores=[]
    for f in fs:
        gain=np.sum((fs[:,0]-f[0])**2+(fs[:,1]-f[1])**2 < 25)
        dist=np.linalg.norm(np.array(f)-np.array(robot))
        scores.append(gain-0.35*dist)
    target=tuple(fs[int(np.argmax(scores))])
    robot=step_toward(robot,target)
    path.append(robot)
path=np.array(path)
reveal(robot)
fs=frontiers()

fig,axes=plt.subplots(1,2,figsize=(13,5))
axes[0].imshow(true,cmap='gray_r',origin='upper'); axes[0].plot(path[:,1],path[:,0],color='#E85D24',lw=2); axes[0].set_title('true map and exploration path')
show=np.zeros((H,W,3)); show[known==-1]=[0.55,0.55,0.55]; show[known==0]=[1,1,1]; show[known==1]=[0,0,0]
axes[1].imshow(show,origin='upper')
if len(fs)>0: axes[1].scatter(fs[:,1],fs[:,0],s=8,color='#1D9E75',label='frontier')
axes[1].plot(path[:,1],path[:,0],color='#E85D24',lw=2,label='robot path'); axes[1].legend(); axes[1].set_title('known map and frontiers')
for ax in axes: ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.savefig('assets/14_frontier_exploration.png',dpi=150,bbox_inches='tight'); plt.show()
print('known ratio:', round(np.mean(known!=-1),3))
print('frontier count:', len(fs))

## 2. Entropy로 보는 정보 획득

탐색 행동의 목표는 불확실성을 줄이는 것이다. Belief entropy가 작아질수록 localization이 더 확실해진다.

In [ ]:
beliefs=[np.ones(8)/8, np.array([0.02,0.03,0.05,0.1,0.5,0.2,0.06,0.04]), np.array([0.005,0.005,0.01,0.02,0.9,0.04,0.01,0.01])]
ent=[-np.sum(b*np.log2(b+1e-12)) for b in beliefs]
fig,axes=plt.subplots(1,3,figsize=(12,3))
for ax,b,h,i in zip(axes,beliefs,ent,range(1,4)):
    ax.bar(np.arange(len(b)),b,color='#534AB7'); ax.set_ylim(0,1); ax.set_title(f'belief {i}: H={h:.2f} bits'); ax.grid(axis='y',alpha=0.2)
plt.tight_layout(); plt.savefig('assets/14_belief_entropy.png',dpi=150,bbox_inches='tight'); plt.show()
print('entropies:', np.round(ent,3))

## 요약

| 개념 | 의미 | 책 커리큘럼 연결 |
|------|------|------------------|
| Frontier | free와 unknown의 경계 | Ch.17 Exploration |
| Information gain | 불확실성 감소량 | active localization |
| Entropy | belief/map uncertainty | exploration objective |
| Trade-off | 정보획득 vs 이동비용 | 실제 탐사 정책 설계 |